In [1]:
# ==================================================
# CELL 1: INSTALL LIBRARIES & LOAD 100% DATA FROM KAGGLEHUB
# ==================================================

import time  # Measure dataset loading time
import pandas as pd  # DataFrame manipulation and tabular data handling
import numpy as np  # Numerical computing and array operations
from sklearn.model_selection import train_test_split  # Split dataset into train, validation, and test sets
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text into TF-IDF feature representations
from sklearn.svm import LinearSVC  # Linear Support Vector Classification model
from sklearn.ensemble import RandomForestClassifier  # Random Forest ensemble classifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix  # Classification evaluation metrics
import xgboost as xgb  # XGBoost gradient boosting library
import kagglehub  # Download datasets directly from Kaggle
from kagglehub import KaggleDatasetAdapter  # Load Kaggle datasets into pandas DataFrames

print("Loading dataset from Kaggle (thedrcat/daigt-v2-train-dataset)...")  # Log start of dataset download
start_time = time.time()  # Record start timestamp

try:  # Use newer dataset_load API if available
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,  # Load data as a pandas DataFrame
        "thedrcat/daigt-v2-train-dataset",  # Kaggle dataset identifier
        "train_v2_drcat_02.csv"  # Target CSV file inside the dataset
    )
except AttributeError:  # Fall back to load_dataset for older kagglehub versions
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,  # Load data as a pandas DataFrame
        "thedrcat/daigt-v2-train-dataset",  # Kaggle dataset identifier
        "train_v2_drcat_02.csv"  # Target CSV file inside the dataset
    )

load_time = time.time() - start_time  # Compute total download and load time in seconds

# Compute raw dataset statistics
total_rows_raw = len(df)  # Total number of rows before cleaning
# Basic data cleaning (drop rows with NaN in text or label columns)
df = df.dropna(subset=['text', 'label'])  # Remove rows missing text or target label
total_rows_clean = len(df)  # Number of valid rows remaining after cleaning

# Calculate percentage of valid data retained
percent_used = (total_rows_clean / total_rows_raw) * 100  # Percentage of raw rows kept

print("=" * 60)  # Print top banner divider
print("DATA USAGE REPORT:")  # Print report title
print(f"- Total initial rows on Kaggle: {total_rows_raw:,} rows")  # Display raw row count
print(f"- Actual rows after cleaning (NaN removed): {total_rows_clean:,} rows")  # Display cleaned row count
print(f"- Usage ratio vs. total raw data: {percent_used:.2f}% (100% of valid data)")  # Display retention percentage
print(f"- Label distribution (Label counts):\n{df['label'].value_counts()}")  # Display class counts
print(f"- Data loading time: {load_time:.2f} seconds")  # Display elapsed loading time
print("=" * 60)  # Print bottom banner divider

# Split into Train / Validation / Test sets (e.g., 80% Train, 10% Val, 10% Test)
X = df['text'].astype(str)  # Input text feature series as strings
y = df['label'].astype(int)  # Target labels as integers

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # Split 80% train and 20% temp stratified by label
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)  # Split temp equally into 10% val and 10% test

print(f"Train set size: {len(X_train):,} rows ({len(X_train)/total_rows_clean*100:.1f}%)")  # Display train size and share
print(f"Validation set size: {len(X_val):,} rows ({len(X_val)/total_rows_clean*100:.1f}%)")  # Display validation size and share
print(f"Test set size: {len(X_test):,} rows ({len(X_test)/total_rows_clean*100:.1f}%)")  # Display test size and share

Loading dataset from Kaggle (thedrcat/daigt-v2-train-dataset)...
Using Colab cache for faster access to the 'daigt-v2-train-dataset' dataset.
DATA USAGE REPORT:
- Total initial rows on Kaggle: 44,868 rows
- Actual rows after cleaning (NaN removed): 44,868 rows
- Usage ratio vs. total raw data: 100.00% (100% of valid data)
- Label distribution (Label counts):
label
0    27371
1    17497
Name: count, dtype: int64
- Data loading time: 8.83 seconds
Train set size: 35,894 rows (80.0%)
Validation set size: 4,487 rows (10.0%)
Test set size: 4,487 rows (10.0%)


In [2]:
# ==================================================
# CELL 2 (OPTIMIZED): TF-IDF FEATURE EXTRACTION (Reduced to 15k features)
# ==================================================

import time  # Measure feature extraction execution time
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text into TF-IDF feature matrices

print("Initializing TfidfVectorizer (Capped at 15,000 features to save RAM)...")  # Log vectorizer setup
start_time = time.time()  # Record start timestamp

# Reduce max_features from 50,000 to 15,000 for faster execution and lower RAM usage
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), stop_words='english')  # Extract 1-2 word n-grams without English stopwords

X_train_tfidf = vectorizer.fit_transform(X_train)  # Fit on train texts and transform train set
X_val_tfidf = vectorizer.transform(X_val)  # Transform validation set using train-fitted vectorizer
X_test_tfidf = vectorizer.transform(X_test)  # Transform test set using train-fitted vectorizer

tfidf_time = time.time() - start_time  # Calculate total vectorization duration in seconds
print(f"Completed TF-IDF extraction in {tfidf_time:.2f} seconds.")  # Print elapsed time
print(f"Train TF-IDF matrix shape: {X_train_tfidf.shape}")  # Print rows and features count

Initializing TfidfVectorizer (Capped at 15,000 features to save RAM)...
Completed TF-IDF extraction in 51.16 seconds.
Train TF-IDF matrix shape: (35894, 15000)


In [3]:
# ==================================================
# CELL 3: TRAIN AND EVALUATE SVM MODEL
# ==================================================

print("Starting SVM (LinearSVC) model training...")  # Log start of SVM training
start_time = time.time()  # Record start timestamp

svm_model = LinearSVC(C=1.0, random_state=42, max_iter=2000)  # Initialize linear SVM with regularization C=1.0 and max 2,000 iterations
svm_model.fit(X_train_tfidf, y_train)  # Train SVM on TF-IDF training features and labels

train_time_svm = time.time() - start_time  # Compute total SVM training duration in seconds

# Evaluate on the test set
y_pred_svm = svm_model.predict(X_test_tfidf)  # Predict class labels on TF-IDF test features
acc_svm = accuracy_score(y_test, y_pred_svm)  # Compute overall test accuracy

print("=" * 60)  # Print top banner divider
print("SVM MODEL RESULTS:")  # Print results header
print(f"- Train time: {train_time_svm:.2f} seconds")  # Display training duration
print(f"- Test Accuracy: {acc_svm * 100:.2f}%")  # Display test accuracy percentage
print("\nClassification Report:")  # Subheader for per-class evaluation metrics
print(classification_report(y_test, y_pred_svm))  # Print precision, recall, and F1-score per class
print("=" * 60)  # Print bottom banner divider

Starting SVM (LinearSVC) model training...
SVM MODEL RESULTS:
- Train time: 0.92 seconds
- Test Accuracy: 99.51%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      2737
           1       1.00      0.99      0.99      1750

    accuracy                           1.00      4487
   macro avg       1.00      0.99      0.99      4487
weighted avg       1.00      1.00      1.00      4487



In [4]:
# ==================================================
# CELL 4: TRAIN AND EVALUATE RANDOM FOREST MODEL
# ==================================================

print("Starting Random Forest model training...")  # Log start of Random Forest training
start_time = time.time()  # Record start timestamp

# Use a moderate n_estimators value to optimize runtime on Colab
rf_model = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)  # Initialize Random Forest with 100 trees, unrestricted depth, and all CPU cores
rf_model.fit(X_train_tfidf, y_train)  # Train Random Forest on TF-IDF training features and labels

train_time_rf = time.time() - start_time  # Compute total Random Forest training duration in seconds

# Evaluate on the test set
y_pred_rf = rf_model.predict(X_test_tfidf)  # Predict class labels on TF-IDF test features
acc_rf = accuracy_score(y_test, y_pred_rf)  # Compute overall test accuracy

print("=" * 60)  # Print top banner divider
print("RANDOM FOREST MODEL RESULTS:")  # Print results header
print(f"- Train time: {train_time_rf:.2f} seconds")  # Display training duration
print(f"- Test Accuracy: {acc_rf * 100:.2f}%")  # Display test accuracy percentage
print("\nClassification Report:")  # Subheader for per-class evaluation metrics
print(classification_report(y_test, y_pred_rf))  # Print precision, recall, and F1-score per class
print("=" * 60)  # Print bottom banner divider

Starting Random Forest model training...
RANDOM FOREST MODEL RESULTS:
- Train time: 59.45 seconds
- Test Accuracy: 98.66%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      2737
           1       0.99      0.98      0.98      1750

    accuracy                           0.99      4487
   macro avg       0.99      0.98      0.99      4487
weighted avg       0.99      0.99      0.99      4487



In [6]:
# ==================================================
# CELL 5 (OPTIMIZED): HIGH-SPEED XGBOOST TRAINING
# ==================================================

import time  # Measure XGBoost training execution time
import xgboost as xgb  # Gradient boosted decision tree library
from sklearn.metrics import classification_report, accuracy_score  # Classification evaluation metrics

print("Starting XGBoost model training (Fast version)...")  # Log start of XGBoost training
start_time = time.time()  # Record start timestamp

# Use tree_method='hist' and cap tree count and depth to train within tens of seconds
xgb_model = xgb.XGBClassifier(
    n_estimators=80,  # Slightly reduce number of trees for faster training
    learning_rate=0.1,  # Step size shrinkage to scale tree contributions
    max_depth=5,  # Cap tree depth to limit RAM consumption and overfitting
    random_state=42,  # Fix random seed for reproducibility
    n_jobs=-1,  # Use all available CPU cores
    tree_method='hist'  # Use histogram-based binning for fast tree construction
)

xgb_model.fit(X_train_tfidf, y_train)  # Train XGBoost classifier on TF-IDF training features and labels

train_time_xgb = time.time() - start_time  # Compute total XGBoost training duration in seconds

# Evaluate on the test set
y_pred_xgb = xgb_model.predict(X_test_tfidf)  # Predict class labels on TF-IDF test features
acc_xgb = accuracy_score(y_test, y_pred_xgb)  # Compute overall test accuracy

print("=" * 60)  # Print top banner divider
print("XGBOOST MODEL RESULTS:")  # Print results header
print(f"- Train time: {train_time_xgb:.2f} seconds")  # Display training duration
print(f"- Test Accuracy: {acc_xgb * 100:.2f}%")  # Display test accuracy percentage
print("=" * 60)  # Print bottom banner divider

Starting XGBoost model training (Fast version)...
XGBOOST MODEL RESULTS:
- Train time: 132.96 seconds
- Test Accuracy: 97.19%


In [ ]:
# ==================================================
# CELL 6A: FINE-TUNE DEEP LEARNING - ELECTRA
# ==================================================

import time  # Measure training and evaluation execution time
import torch  # PyTorch deep learning framework
import numpy as np  # Numerical operations and array handling
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments  # Hugging Face model and Trainer utilities
from datasets import Dataset  # Memory-efficient Hugging Face dataset wrapper
from sklearn.metrics import accuracy_score, precision_recall_fscore_support  # Classification evaluation metrics

# 1. Prepare Hugging Face Dataset objects from existing pandas variables
print("Preparing Dataset from Pandas...")  # Log dataset conversion start
train_df = pd.DataFrame({'text': X_train, 'label': y_train})  # Combine training texts and labels into a DataFrame
val_df = pd.DataFrame({'text': X_val, 'label': y_val})  # Combine validation texts and labels into a DataFrame

train_dataset = Dataset.from_pandas(train_df)  # Convert training DataFrame to Hugging Face Dataset
val_dataset = Dataset.from_pandas(val_df)  # Convert validation DataFrame to Hugging Face Dataset

# 2. Load Tokenizer and Model
MODEL_NAME_ELECTRA = "google/electra-base-discriminator"  # Pretrained ELECTRA base discriminator checkpoint
print(f"Loading Tokenizer and Model: {MODEL_NAME_ELECTRA}...")  # Log model checkpoint download
tokenizer_electra = AutoTokenizer.from_pretrained(MODEL_NAME_ELECTRA)  # Load subword tokenizer for ELECTRA

def tokenize_function_electra(examples):  # Tokenize a batch of text examples
    return tokenizer_electra(examples['text'], padding="max_length", truncation=True, max_length=256)  # Pad/truncate sequences to length 256

print("Tokenizing data for ELECTRA...")  # Log tokenization progress
tokenized_train_electra = train_dataset.map(tokenize_function_electra, batched=True)  # Tokenize training dataset in batches
tokenized_val_electra = val_dataset.map(tokenize_function_electra, batched=True)  # Tokenize validation dataset in batches

model_electra = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_ELECTRA, num_labels=2)  # Load ELECTRA with a 2-class classification head

# 3. Define evaluation metrics function
def compute_metrics(eval_pred):  # Compute classification metrics from logits and ground-truth labels
    predictions, labels = eval_pred  # Unpack raw model predictions (logits) and true labels
    preds = np.argmax(predictions, axis=1)  # Select class index with highest logit
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')  # Compute binary precision, recall, and F1-score
    acc = accuracy_score(labels, preds)  # Compute overall classification accuracy
    return {  # Return dictionary of metrics for Trainer
        'accuracy': acc,  # Overall accuracy
        'f1': f1,  # Binary F1-score
        'precision': precision,  # Binary precision
        'recall': recall  # Binary recall
    }

# 4. Configure training hyperparameters
training_args_electra = TrainingArguments(  # Set Trainer configuration
    output_dir="./results_electra",  # Directory to save checkpoints and outputs
    eval_strategy="epoch",  # Evaluate on validation set at the end of each epoch
    save_strategy="epoch",  # Save model checkpoint at the end of each epoch
    learning_rate=2e-5,  # Learning rate for AdamW optimizer
    per_device_train_batch_size=16,  # Training batch size per GPU/device
    per_device_eval_batch_size=16,  # Evaluation batch size per GPU/device
    num_train_epochs=2,  # Number of full passes over the training set
    weight_decay=0.01,  # Weight decay regularization factor
    load_best_model_at_end=True,  # Restore best checkpoint at end of training
    metric_for_best_model="accuracy",  # Select best model based on validation accuracy
    logging_steps=100,  # Log training metrics every 100 steps
    fp16=torch.cuda.is_available()  # Enable mixed precision (fp16) when CUDA GPU is available
)

trainer_electra = Trainer(  # Initialize Hugging Face Trainer for ELECTRA
    model=model_electra,  # Sequence classification model
    args=training_args_electra,  # Training arguments
    train_dataset=tokenized_train_electra,  # Tokenized training dataset
    eval_dataset=tokenized_val_electra,  # Tokenized validation dataset
    compute_metrics=compute_metrics,  # Custom evaluation metric function
)

print(f"Starting fine-tuning for {MODEL_NAME_ELECTRA}...")  # Log start of fine-tuning
start_time_electra = time.time()  # Record start timestamp
trainer_electra.train()  # Run fine-tuning loop
train_time_electra = time.time() - start_time_electra  # Calculate total training duration in seconds

eval_results_electra = trainer_electra.evaluate()  # Evaluate best ELECTRA checkpoint on validation set
print("=" * 60)  # Print top banner divider
print(f"RESULTS FOR {MODEL_NAME_ELECTRA}:")  # Print model results header
print(f"- Train time: {train_time_electra:.2f} seconds (~{train_time_electra/60:.1f} minutes)")  # Display training time in seconds and minutes
print(f"- Validation evaluation: {eval_results_electra}")  # Display validation metric dictionary
print("=" * 60)  # Print bottom banner divider

Preparing Dataset from Pandas...
Loading Tokenizer and Model: google/electra-base-discriminator...


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing data for ELECTRA...


... (7 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Starting fine-tuning for google/electra-base-discriminator...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.027378,0.048666,0.991085,0.988656,0.981419,0.996000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ==================================================
# CELL 6B: FINE-TUNE DEEP LEARNING - ROBERTA (roberta-base)
# ==================================================

import time  # Measure training and evaluation execution time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments  # Hugging Face model and Trainer utilities

MODEL_NAME_ROBERTA = "roberta-base"  # Pretrained RoBERTa base checkpoint

print(f"Loading Tokenizer and Model: {MODEL_NAME_ROBERTA}...")  # Log model checkpoint loading
tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_NAME_ROBERTA)  # Load subword tokenizer for RoBERTa

def tokenize_function_roberta(examples):  # Tokenize a batch of text examples
    return tokenizer_roberta(examples['text'], padding="max_length", truncation=True, max_length=256)  # Pad or truncate sequences to max length 256

print("Tokenizing data for RoBERTa...")  # Log tokenization progress
tokenized_train_roberta = train_dataset.map(tokenize_function_roberta, batched=True)  # Tokenize training dataset in batches
tokenized_val_roberta = val_dataset.map(tokenize_function_roberta, batched=True)  # Tokenize validation dataset in batches

model_roberta = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_ROBERTA, num_labels=2)  # Load RoBERTa with a 2-class classification head

training_args_roberta = TrainingArguments(  # Configure training hyperparameters for Trainer
    output_dir="./results_roberta",  # Directory to store checkpoints and logs
    eval_strategy="epoch",  # Evaluate on validation set after each epoch
    save_strategy="epoch",  # Save model checkpoint after each epoch
    learning_rate=2e-5,  # Learning rate for AdamW optimizer
    per_device_train_batch_size=16,  # Training batch size per device
    per_device_eval_batch_size=16,  # Evaluation batch size per device
    num_train_epochs=2,  # Number of training epochs
    weight_decay=0.01,  # Weight decay regularization
    load_best_model_at_end=True,  # Reload best checkpoint at the end of training
    metric_for_best_model="accuracy",  # Select best checkpoint based on validation accuracy
    logging_steps=100,  # Log training progress every 100 steps
    fp16=torch.cuda.is_available()  # Enable mixed precision (fp16) when CUDA GPU is available
)

trainer_roberta = Trainer(  # Initialize Hugging Face Trainer for RoBERTa
    model=model_roberta,  # Sequence classification model
    args=training_args_roberta,  # Training configuration
    train_dataset=tokenized_train_roberta,  # Tokenized training dataset
    eval_dataset=tokenized_val_roberta,  # Tokenized validation dataset
    compute_metrics=compute_metrics,  # Evaluation metric function
)

print(f"Starting fine-tuning for {MODEL_NAME_ROBERTA}...")  # Log start of fine-tuning
start_time_roberta = time.time()  # Record start timestamp
trainer_roberta.train()  # Run fine-tuning loop
train_time_roberta = time.time() - start_time_roberta  # Calculate total training duration in seconds

eval_results_roberta = trainer_roberta.evaluate()  # Evaluate best RoBERTa checkpoint on validation set
print("=" * 60)  # Print top banner divider
print(f"RESULTS FOR {MODEL_NAME_ROBERTA}:")  # Print model results header
print(f"- Train time: {train_time_roberta:.2f} seconds (~{train_time_roberta/60:.1f} minutes)")  # Display training time in seconds and minutes
print(f"- Validation evaluation: {eval_results_roberta}")  # Display validation metric dictionary
print("=" * 60)  # Print bottom banner divider

In [ ]:
# ==================================================
# CELL 7: SUMMARY AND PERFORMANCE COMPARISON TABLE ACROSS ALL MODELS
# ==================================================

import pandas as pd  # Library for constructing and displaying tabular summaries

# Convert duration in seconds into a human-readable string (seconds or minutes + seconds)
def format_time(seconds):  # Format elapsed time in seconds
    if seconds < 60:  # Display directly in seconds for runs under one minute
        return f"{seconds:.2f} seconds"  # Format up to two decimal places
    else:  # Display minutes along with total seconds for longer runs
        return f"{seconds/60:.2f} minutes ({seconds:.1f}s)"  # Format minutes and seconds

summary_data = [  # Rows comparing all evaluated baseline and deep learning models
    {
        "Model": "SVM (LinearSVC)",  # Linear Support Vector Machine classifier
        "Method Type": "Traditional ML",  # Model paradigm
        "Features": "TF-IDF (15k)",  # Input feature representation
        "Data Ratio": f"{percent_used:.1f}%",  # Percentage of dataset used
        "Accuracy (Acc)": f"{acc_svm*100:.2f}%",  # Test accuracy percentage
        "Train Time": format_time(train_time_svm)  # Formatted SVM training duration
    },
    {
        "Model": "Random Forest",  # Random Forest ensemble classifier
        "Method Type": "Traditional ML",  # Model paradigm
        "Features": "TF-IDF (15k)",  # Input feature representation
        "Data Ratio": f"{percent_used:.1f}%",  # Percentage of dataset used
        "Accuracy (Acc)": f"{acc_rf*100:.2f}%",  # Test accuracy percentage
        "Train Time": format_time(train_time_rf)  # Formatted Random Forest training duration
    },
    {
        "Model": "XGBoost",  # Gradient boosted decision tree classifier
        "Method Type": "Traditional ML",  # Model paradigm
        "Features": "TF-IDF (15k)",  # Input feature representation
        "Data Ratio": f"{percent_used:.1f}%",  # Percentage of dataset used
        "Accuracy (Acc)": f"{acc_xgb*100:.2f}%",  # Test accuracy percentage
        "Train Time": format_time(train_time_xgb)  # Formatted XGBoost training duration
    },
    {
        "Model": "RoBERTa (roberta-base)",  # Pretrained RoBERTa base sequence classifier
        "Method Type": "Deep Learning",  # Model paradigm
        "Features": "Token Embeddings",  # Contextual subword token representations
        "Data Ratio": f"{percent_used:.1f}%",  # Percentage of dataset used
        "Accuracy (Acc)": f"{eval_results_roberta.get('eval_accuracy', 0)*100:.2f}%",  # Validation accuracy percentage
        "Train Time": format_time(train_time_roberta)  # Formatted RoBERTa training duration
    },
    {
        "Model": "ELECTRA",  # Pretrained ELECTRA base discriminator classifier
        "Method Type": "Deep Learning",  # Model paradigm
        "Features": "Token Embeddings",  # Contextual subword token representations
        "Data Ratio": f"{percent_used:.1f}%",  # Percentage of dataset used
        "Accuracy (Acc)": f"{eval_results_electra.get('eval_accuracy', 0)*100:.2f}%",  # Validation accuracy percentage
        "Train Time": format_time(train_time_electra)  # Formatted ELECTRA training duration
    }
]

df_summary = pd.DataFrame(summary_data)  # Convert comparison records into a pandas DataFrame

print("=" * 105)  # Top banner border
print("EXPERIMENT SUMMARY: ACCURACY AND RUNTIME COMPARISON")  # Table header title
print("=" * 105)  # Bottom banner border
display(df_summary)  # Render summary DataFrame in notebook output
print("=" * 105)  # Closing border line